# Physics-Informed Conditional Invertible Neural Network (PI-cINN)
## Architecture Reference — Underwater Acoustic Metamaterial Inverse Design

---

| Property | Value |
|---|---|
| **Model** | PI-cINN (Conditional Normalizing Flow) |
| **Task** | Inverse design: target absorption spectrum → metamaterial parameters |
| **Parameter space** | 20-dimensional (d₁–d₁₀, m₂,m₃,m₅,m₆,m₈,m₉, ρ, η, E, ν) |
| **Condition space** | 1000-point absorption spectrum (200–2000 Hz) |
| **Latent space** | 20-dimensional standard Gaussian z ~ N(0, I) |
| **Total parameters** | 1,207,072 (~1.2M) |
| **Training loss** | L_total = L_NLL + λ · L_physics |
| **Physics engine** | Differentiable Transfer Matrix Method (TMM), 10-layer stack |
| **Dataset** | 1,000,000 LHS-sampled configurations |

## Figure 1 — Complete PI-cINN System Overview

The master diagram below shows the full architecture including both training paths (NLL + Physics), inference, and backpropagation gradient flow. All dimensions, component names, and parameter counts are labeled for direct replication in a conference paper figure.

In [ ]:
"""
┌──────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                          PHYSICS-INFORMED cINN — COMPLETE SYSTEM OVERVIEW                                │
│                                      (1,207,072 trainable parameters)                                    │
│                                                                                                          │
│   ╔══════════════════════════════════════════════════════════════════════════════════════════════════╗     │
│   ║                                   TRAINING  PHASE                                               ║     │
│   ╚══════════════════════════════════════════════════════════════════════════════════════════════════╝     │
│                                                                                                          │
│   INPUTS                                                                                                 │
│   ──────                                                                                                 │
│   x_params ∈ ℝ^(B×20)  ─────────────────────────────────────────────────────┐                            │
│   (scaled design parameters)                                                 │                            │
│                                                                              │                            │
│   y_spectra ∈ ℝ^(B×1000) ──┐                                                │                            │
│   (absorption spectrum)     │                                                │                            │
│                             ▼                                                │                            │
│                  ┌─────────────────────┐                                     │                            │
│                  │   SPECTRUM ENCODER   │                                     │                            │
│                  │   ─────────────────  │                                     │                            │
│                  │   Linear 1000→256    │                                     │                            │
│                  │   LeakyReLU(0.01)    │                                     │                            │
│                  │   Linear 256→256     │                                     │                            │
│                  │   LeakyReLU(0.01)    │                                     │                            │
│                  │   Linear 256→128     │                                     │                            │
│                  │   (354,944 params)   │                                     │                            │
│                  └──────────┬──────────┘                                     │                            │
│                             │                                                │                            │
│                        cond ∈ ℝ^(B×128)                                      │                            │
│                             │                                                │                            │
│      ┌──────────────────────┼────────────────────────────────────────────┐    │                            │
│      │                      │           FLOW CHAIN (8 BLOCKS)            │    │                            │
│      │                      │           ─────────────────────            │    │                            │
│      │                      │     ┌─────────────────────────────────┐    │    │                            │
│      │                      ├────►│  COUPLING BLOCK 1 (106,516 p)   │    │    │                            │
│      │                      │     │  Split → SubNet → Affine → Flip │    │    │                            │
│      │                      │     └────────────────┬────────────────┘    │    │                            │
│      │                      │     ┌────────────────┴────────────────┐    │    │                            │
│      │                      ├────►│  COUPLING BLOCK 2 (106,516 p)   │    │    │                            │
│      │                      │     └────────────────┬────────────────┘    │    │                            │
│      │                      │                      ⋮                     │    │                            │
│      │                      │     ┌────────────────┴────────────────┐    │    │                            │
│      │                      ├────►│  COUPLING BLOCK 7 (106,516 p)   │    │    │                            │
│      │                      │     └────────────────┬────────────────┘    │    │                            │
│      │                      │     ┌────────────────┴────────────────┐    │    │                            │
│      │                      └────►│  COUPLING BLOCK 8 (106,516 p)   │◄───┼────┘  x_params                 │
│      │                            └────────────────┬────────────────┘    │        enters here              │
│      │                              8 × 106,516 = 852,128 params         │                                │
│      └──────────────────────────────────┼───────────────────────────────┘                                 │
│                                         │                                                                 │
│                              ┌──────────┴──────────┐                                                      │
│                              │                      │                                                      │
│                       z ∈ ℝ^(B×20)          log_det_J ∈ ℝ^B                                               │
│                              │                      │                                                      │
│   ═══════════════════════════╪══════════════════════╪══════════════════════════════════                    │
│        PATH A: NLL LOSS      │                      │                                                      │
│   ═══════════════════════════╪══════════════════════╪══════════════════════════════════                    │
│                              │                      │                                                      │
│                              ▼                      ▼                                                      │
│                     ┌──────────────────────────────────────┐                                               │
│                     │           NLL LOSS (L_NLL)            │                                               │
│                     │                                       │                                               │
│                     │   L_NLL = mean( 0.5·‖z‖² − log|J| )  │                                               │
│                     │                                       │                                               │
│                     │   Enforces: latent space → N(0,I)     │                                               │
│                     └──────────────────┬───────────────────┘                                               │
│                                        │                                                                   │
│   ═════════════════════════════════════╪═══════════════════════════════════════════════                    │
│        PATH B: PHYSICS LOSS (λ=0.1)   │    [active after warmup epoch 20]                                  │
│   ═════════════════════════════════════╪═══════════════════════════════════════════════                    │
│                                        │                                                                   │
│   z_sample ~ N(0,I) ∈ ℝ^(64×20)       │                                                                   │
│        │                               │                                                                   │
│        ▼                               │                                                                   │
│   ┌─────────────────────────┐          │                                                                   │
│   │  cINN REVERSE PASS      │          │                                                                   │
│   │  (8 blocks, reversed)   │◄── cond  │                                                                   │
│   │  Un-flip → Inv. Affine  │          │                                                                   │
│   └────────────┬────────────┘          │                                                                   │
│                │                       │                                                                   │
│         x_gen ∈ ℝ^(64×20)             │                                                                   │
│                │                       │                                                                   │
│                ▼                       │                                                                   │
│   ┌──────────────────────────┐         │                                                                   │
│   │  DIFFERENTIABLE TMM      │         │                                                                   │
│   │  ──────────────────────  │         │                                                                   │
│   │  Inverse StandardScaler  │         │                                                                   │
│   │  Soft-clamp to bounds    │         │                                                                   │
│   │  10-layer acoustic TMM   │         │                                                                   │
│   │  (GPU, pure PyTorch)     │         │                                                                   │
│   └────────────┬─────────────┘         │                                                                   │
│                │                       │                                                                   │
│         α_pred ∈ ℝ^(64×1000)          │                                                                   │
│                │                       │                                                                   │
│                ▼                       │                                                                   │
│   ┌──────────────────────────┐         │                                                                   │
│   │     PHYSICS LOSS         │         │                                                                   │
│   │                          │         │                                                                   │
│   │  L_phys = MSE(α_pred,   │         │                                                                   │
│   │              y_spectra)  │         │                                                                   │
│   │                          │         │                                                                   │
│   │  Enforces: generated     │         │                                                                   │
│   │  params → valid spectra  │         │                                                                   │
│   └────────────┬─────────────┘         │                                                                   │
│                │                       │                                                                   │
│   ═════════════╪═══════════════════════╪═══════════════════════════════════════════════                    │
│        COMBINED LOSS                   │                                                                   │
│   ═════════════╪═══════════════════════╪═══════════════════════════════════════════════                    │
│                │                       │                                                                   │
│                └───────────┬───────────┘                                                                   │
│                            ▼                                                                               │
│                ┌───────────────────────────┐                                                               │
│                │  L_total = L_NLL + λ·L_phys│                                                               │
│                └─────────────┬─────────────┘                                                               │
│                              │                                                                             │
│                              ▼                                                                             │
│                    ∂L_total / ∂θ  (backpropagation)                                                        │
│                              │                                                                             │
│                  ┌───────────┴───────────┐                                                                 │
│                  ▼                       ▼                                                                  │
│          ∂L/∂(SubNet W)         ∂L/∂(Encoder W)                                                            │
│          8×3 Linear layers      3 Linear layers                                                            │
│                  │                       │                                                                  │
│                  └───────────┬───────────┘                                                                 │
│                              ▼                                                                             │
│                  ┌─────────────────────────┐                                                               │
│                  │  ADAM OPTIMIZER           │                                                               │
│                  │  lr = 1e-3               │                                                               │
│                  │  weight_decay = 1e-5     │                                                               │
│                  │  grad_clip = 1.0         │                                                               │
│                  │  CosineAnnealing(T=250)  │                                                               │
│                  └─────────────────────────┘                                                               │
│                                                                                                            │
│   ╔══════════════════════════════════════════════════════════════════════════════════════════════════╗     │
│   ║                                  INFERENCE  PHASE                                               ║     │
│   ╚══════════════════════════════════════════════════════════════════════════════════════════════════╝     │
│                                                                                                          │
│   y_target ∈ ℝ^(1×1000) ──► Encoder ──► cond                                                            │
│                                            │                                                              │
│   z ~ N(0,I) ∈ ℝ^(N×20) ──────────────────┤                                                              │
│                                            ▼                                                              │
│                                   cINN Reverse Pass                                                       │
│                                            │                                                              │
│                                   x_scaled ∈ ℝ^(N×20)                                                    │
│                                            │                                                              │
│                                   Inverse Scaler → Physical units                                         │
│                                            │                                                              │
│                                   Clip to valid bounds                                                    │
│                                            │                                                              │
│                                   TMM Forward Simulation                                                  │
│                                            │                                                              │
│                                   Select best by min RMSE                                                 │
│                                                                                                          │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────┘
"""

## Figure 2 — Network Component Details and Parameter Counts

In [ ]:
"""
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│                              SPECTRUM ENCODER                                            │
│                                                                                          │
│   Function: Compress 1000-dim absorption spectrum into 128-dim condition embedding       │
│                                                                                          │
│   y_spectra ∈ ℝ^(B×1000)                                                                │
│        │                                                                                 │
│        ▼                                                                                 │
│   ┌────────────────────────────────────────────────────────────────────────┐              │
│   │  Layer                │ Shape          │ Weights   │ Biases │ Total   │              │
│   │  ─────────────────────┼────────────────┼───────────┼────────┼─────────│              │
│   │  Linear_1             │ 1000 → 256     │ 256,000   │ 256    │ 256,256 │              │
│   │  LeakyReLU(0.01)      │ —              │ 0         │ 0      │ 0       │              │
│   │  Linear_2             │ 256 → 256      │ 65,536    │ 256    │ 65,792  │              │
│   │  LeakyReLU(0.01)      │ —              │ 0         │ 0      │ 0       │              │
│   │  Linear_3             │ 256 → 128      │ 32,768    │ 128    │ 32,896  │              │
│   │  ─────────────────────┼────────────────┼───────────┼────────┼─────────│              │
│   │  SUBTOTAL             │                │ 354,304   │ 640    │ 354,944 │              │
│   └────────────────────────────────────────────────────────────────────────┘              │
│        │                                                                                 │
│        ▼                                                                                 │
│   cond ∈ ℝ^(B×128)                                                                      │
│                                                                                          │
│─────────────────────────────────────────────────────────────────────────────────────────│
│                                                                                          │
│                     SINGLE COUPLING BLOCK SUBNET                                         │
│                                                                                          │
│   Function: Predict affine parameters (scale s, translation t) from                      │
│             unchanged half x₁ concatenated with condition embedding                      │
│                                                                                          │
│   [x₁ ∈ ℝ^10 ; cond ∈ ℝ^128] = input ∈ ℝ^138                                          │
│        │                                                                                 │
│        ▼                                                                                 │
│   ┌────────────────────────────────────────────────────────────────────────┐              │
│   │  Layer                │ Shape          │ Weights   │ Biases │ Total   │              │
│   │  ─────────────────────┼────────────────┼───────────┼────────┼─────────│              │
│   │  Linear_1             │ 138 → 256      │ 35,328    │ 256    │ 35,584  │              │
│   │  LeakyReLU(0.01)      │ —              │ 0         │ 0      │ 0       │              │
│   │  Linear_2             │ 256 → 256      │ 65,536    │ 256    │ 65,792  │              │
│   │  LeakyReLU(0.01)      │ —              │ 0         │ 0      │ 0       │              │
│   │  Linear_3             │ 256 → 20       │ 5,120     │ 20     │ 5,140   │              │
│   │  ─────────────────────┼────────────────┼───────────┼────────┼─────────│              │
│   │  SUBTOTAL (per block) │                │ 105,984   │ 532    │ 106,516 │              │
│   └────────────────────────────────────────────────────────────────────────┘              │
│        │                                                                                 │
│        ▼                                                                                 │
│   [s ∈ ℝ^10 ; t ∈ ℝ^10] = output ∈ ℝ^20                                                │
│                                                                                          │
│─────────────────────────────────────────────────────────────────────────────────────────│
│                                                                                          │
│                       PARAMETER COUNT SUMMARY                                            │
│                                                                                          │
│   ┌─────────────────────────────┬────────────────┬───────────────────┐                   │
│   │  Component                  │ Count          │ Parameters        │                   │
│   │  ───────────────────────────┼────────────────┼───────────────────│                   │
│   │  SpectrumEncoder            │ 1              │ 354,944           │                   │
│   │  Coupling Block SubNets     │ 8 × 106,516   │ 852,128           │                   │
│   │  ───────────────────────────┼────────────────┼───────────────────│                   │
│   │  GRAND TOTAL                │                │ 1,207,072 (~1.2M) │                   │
│   └─────────────────────────────┴────────────────┴───────────────────┘                   │
│                                                                                          │
│   Note: Coupling block operations (exp, affine, flip) have ZERO learnable parameters.    │
│         All learning occurs in SubNets (predict s, t) and Encoder (predict cond).        │
│                                                                                          │
└─────────────────────────────────────────────────────────────────────────────────────────┘
"""

## Figure 3 — Forward Pass: Detailed Data Flow Through a Single Coupling Block

During training, design parameters **x** flow forward through 8 coupling blocks to produce latent vector **z** and log-determinant of the Jacobian. Each block splits the 20-dim vector into two halves, transforms one half conditioned on the other, and swaps them.

In [ ]:
"""
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                    FORWARD PASS — SINGLE COUPLING BLOCK (DETAIL)                         │
│                                                                                          │
│   x ∈ ℝ^(B×20)          cond ∈ ℝ^(B×128)                                                │
│       │                       │                                                          │
│       │    SPLIT              │                                                          │
│       ├──────────┐            │                                                          │
│       │          │            │                                                          │
│       ▼          ▼            │                                                          │
│   x₁ = x[:,:10]  x₂ = x[:,10:]                                                         │
│   (B×10)          (B×10)      │                                                          │
│       │              │        │                                                          │
│       │              │        │                                                          │
│       │    CONCATENATE        │                                                          │
│       ├───────────────────────┤                                                          │
│       │                       │                                                          │
│       ▼                       │                                                          │
│   [x₁ ; cond] ∈ ℝ^(B×138)   │                                                          │
│       │                       │                                                          │
│       ▼                       │                                                          │
│   ┌─────────────────────┐     │                                                          │
│   │      SubNet          │     │                                                          │
│   │  138→256→256→20      │     │                                                          │
│   │  (106,516 params)    │     │                                                          │
│   └──────────┬──────────┘     │                                                          │
│              │                │                                                          │
│         st ∈ ℝ^(B×20)        │                                                          │
│              │                │                                                          │
│       ┌──────┴──────┐        │                                                          │
│       │             │        │                                                          │
│       ▼             ▼        │                                                          │
│    s = st[:,:10]  t = st[:,10:]                                                          │
│    (B×10)         (B×10)     │                                                          │
│       │                      │                                                          │
│       ▼                      │                                                          │
│    s = soft_clamp(s, −3, 3)  │   ◄── differentiable sigmoid-based clamp                 │
│       │                      │                                                          │
│       │     AFFINE TRANSFORM │                                                          │
│       │          ┌───────────┘                                                          │
│       │          │                                                                       │
│       ▼          ▼                                                                       │
│    y₂ = x₂ · exp(s) + t        ◄── invertible affine coupling                           │
│    (B×10)                                                                                │
│       │                                                                                  │
│       │     ACCUMULATE JACOBIAN                                                          │
│       │                                                                                  │
│    log_det_J += Σ(s, dim=1)     ◄── log|det(J)| for this block                          │
│       │                                                                                  │
│       │     RECOMBINE                                                                    │
│       │                                                                                  │
│    output = cat([x₁, y₂])  ∈ ℝ^(B×20)                                                  │
│       │                                                                                  │
│       │     FLIP (swap halves for next block)                                            │
│       │                                                                                  │
│    output = output[:, [10:20, 0:10]]  ∈ ℝ^(B×20)                                        │
│       │                                                                                  │
│       ▼                                                                                  │
│   → next coupling block                                                                  │
│                                                                                          │
│   After 8 blocks:                                                                        │
│       z ∈ ℝ^(B×20)  ,   total_log_det_J ∈ ℝ^B                                           │
│                                                                                          │
└──────────────────────────────────────────────────────────────────────────────────────────┘


┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                    FORWARD PASS — FULL 8-BLOCK CHAIN                                     │
│                                                                                          │
│   x_params ∈ ℝ^(B×20)     y_spectra ∈ ℝ^(B×1000) ──► Encoder ──► cond ∈ ℝ^(B×128)     │
│       │                                                          │                       │
│       ▼                                                          │                       │
│   ┌──────────────────────────────────────────────────────────────┼──────────────────┐    │
│   │  Block 1:  x₁=[0:10] identity  │  x₂=[10:20] transformed   │  → FLIP          │    │
│   ├──────────────────────────────────────────────────────────────┼──────────────────┤    │
│   │  Block 2:  x₁=[10:20] identity │  x₂=[0:10]  transformed   │  → FLIP          │    │
│   ├──────────────────────────────────────────────────────────────┼──────────────────┤    │
│   │  Block 3:  x₁=[0:10] identity  │  x₂=[10:20] transformed   │  → FLIP          │    │
│   ├──────────────────────────────────────────────────────────────┼──────────────────┤    │
│   │  Block 4:  x₁=[10:20] identity │  x₂=[0:10]  transformed   │  → FLIP          │    │
│   ├──────────────────────────────────────────────────────────────┼──────────────────┤    │
│   │  Block 5:  x₁=[0:10] identity  │  x₂=[10:20] transformed   │  → FLIP          │    │
│   ├──────────────────────────────────────────────────────────────┼──────────────────┤    │
│   │  Block 6:  x₁=[10:20] identity │  x₂=[0:10]  transformed   │  → FLIP          │    │
│   ├──────────────────────────────────────────────────────────────┼──────────────────┤    │
│   │  Block 7:  x₁=[0:10] identity  │  x₂=[10:20] transformed   │  → FLIP          │    │
│   ├──────────────────────────────────────────────────────────────┼──────────────────┤    │
│   │  Block 8:  x₁=[10:20] identity │  x₂=[0:10]  transformed   │  → FLIP          │    │
│   └──────────────────────────────────────────────────────────────┴──────────────────┘    │
│       │                                                                                  │
│       ▼                                                                                  │
│   z ∈ ℝ^(B×20)     log_det_J ∈ ℝ^B                                                      │
│                                                                                          │
│   Dims [0:10]  transformed in blocks: 2, 4, 6, 8  (4 times each)                        │
│   Dims [10:20] transformed in blocks: 1, 3, 5, 7  (4 times each)                        │
│   ✓ SYMMETRIC — every dimension transformed equally                                      │
│                                                                                          │
└──────────────────────────────────────────────────────────────────────────────────────────┘
"""

## Figure 4 — Reverse Pass (Inference): Latent Sample → Physical Parameters → Spectrum

During inference, latent samples **z ~ N(0, I)** are pushed backward through the 8 coupling blocks in reverse order. Each block applies the **inverse** affine transform. The output is then unscaled, clipped to physical bounds, and validated through the TMM physics engine.

In [ ]:
"""
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                    REVERSE PASS — INFERENCE PIPELINE                                     │
│                                                                                          │
│   STEP 1: ENCODE TARGET SPECTRUM                                                         │
│   ─────────────────────────────                                                          │
│                                                                                          │
│   y_target ∈ ℝ^(1×1000)                                                                 │
│   (desired absorption spectrum)                                                          │
│        │                                                                                 │
│        ▼                                                                                 │
│   ┌─────────────────────┐                                                                │
│   │   SPECTRUM ENCODER   │  ◄── same trained weights as training phase                   │
│   │   1000 → 256 → 256   │                                                               │
│   │   → 128              │                                                               │
│   └──────────┬──────────┘                                                                │
│              │                                                                           │
│         cond ∈ ℝ^(1×128)                                                                 │
│              │                                                                           │
│                                                                                          │
│   STEP 2: SAMPLE LATENT SPACE                                                            │
│   ────────────────────────────                                                           │
│                                                                                          │
│   z ~ N(0, I) ∈ ℝ^(N×20)      (N = number of candidate solutions, e.g., 5000)           │
│        │                                                                                 │
│        │                                                                                 │
│   STEP 3: REVERSE FLOW (8 blocks in reverse order: block 8 → block 1)                   │
│   ─────────────────────────────────────────────────────────────────────                   │
│        │                                                                                 │
│        ▼                                                                                 │
│   For each block i = 8, 7, 6, 5, 4, 3, 2, 1:                                            │
│                                                                                          │
│      ┌──────────────────────────────────────────────────────────────────────────┐         │
│      │  STEP 3a: UN-FLIP                                                        │         │
│      │     x = x[:, [10:20, 0:10]]         (swap halves back)                   │         │
│      │                                                                          │         │
│      │  STEP 3b: SPLIT                                                          │         │
│      │     x₁ = x[:, :10]                  (unchanged half)                     │         │
│      │     x₂ = x[:, 10:]                  (previously transformed half)        │         │
│      │                                                                          │         │
│      │  STEP 3c: COMPUTE AFFINE PARAMETERS                                     │         │
│      │     [x₁ ; cond] → SubNet_i → s, t                                       │         │
│      │     s = soft_clamp(s, −3, 3)                                             │         │
│      │                                                                          │         │
│      │  STEP 3d: INVERSE AFFINE TRANSFORM                                      │         │
│      │     x₂_recovered = (x₂ − t) · exp(−s)     ◄── exact inverse             │         │
│      │                                                                          │         │
│      │  STEP 3e: RECOMBINE                                                      │         │
│      │     output = cat([x₁, x₂_recovered])                                    │         │
│      └──────────────────────────────────────────────────────────────────────────┘         │
│        │                                                                                 │
│        ▼                                                                                 │
│   x_scaled ∈ ℝ^(N×20)   (StandardScaler space)                                          │
│        │                                                                                 │
│                                                                                          │
│   STEP 4: CONVERT TO PHYSICAL UNITS                                                      │
│   ─────────────────────────────────                                                      │
│        │                                                                                 │
│        ▼                                                                                 │
│   x_physical = x_scaled · σ + μ           (inverse StandardScaler)                       │
│        │                                                                                 │
│        ▼                                                                                 │
│   ┌─────────────────────────────────────────────────────────────────────────┐             │
│   │  VALIDATE AND CLIP TO PHYSICAL BOUNDS                                   │             │
│   │                                                                         │             │
│   │  d₁–d₁₀    ∈ [1×10⁻³, 20×10⁻³] m     (layer thicknesses)              │             │
│   │  m₂,₃,₅,₆,₈,₉ ∈ [20×10⁻³, 1980×10⁻³] m  (hollow cavity dims)        │             │
│   │  ρ         ∈ [1000, 1500] kg/m³        (density)                        │             │
│   │  η         ∈ [0.1, 0.8]               (loss factor)                     │             │
│   │  E         ∈ [E_min(η), E_max(η)]     (Young's modulus, η-dependent)    │             │
│   │  ν         ∈ [0.4, 0.49]              (Poisson's ratio)                 │             │
│   │                                                                         │             │
│   │  E_min = 1×10⁷·(1+η),  E_max = 1×10⁸·(1+η)                            │             │
│   └─────────────────────────────────────────────────────────────────────────┘             │
│        │                                                                                 │
│        ▼                                                                                 │
│   x_physical ∈ ℝ^(N×20)   (valid metamaterial configurations)                            │
│        │                                                                                 │
│                                                                                          │
│   STEP 5: PHYSICS VALIDATION VIA TMM                                                     │
│   ───────────────────────────────────                                                    │
│        │                                                                                 │
│        ▼                                                                                 │
│   ┌──────────────────────────────────────────────────────┐                                │
│   │  TRANSFER MATRIX METHOD (TMM)                        │                                │
│   │  ─────────────────────────────                       │                                │
│   │  For each of N candidates:                           │                                │
│   │    • Build 10-layer impedance stack                  │                                │
│   │    • Compute complex transfer matrices               │                                │
│   │    • Calculate absorption at 1000 frequencies        │                                │
│   │    • α_pred ∈ ℝ^(N×1000)                            │                                │
│   └──────────────────────┬───────────────────────────────┘                                │
│                          │                                                                │
│                          ▼                                                                │
│   STEP 6: CANDIDATE SELECTION                                                             │
│   ───────────────────────────                                                             │
│                                                                                          │
│   RMSE_i = √( mean( (α_pred_i − y_target)² ) )    for i = 1…N                           │
│                                                                                          │
│   best = argmin(RMSE)                                                                     │
│                                                                                          │
│   OUTPUT: x_physical[best] ∈ ℝ^20                                                        │
│           (optimal metamaterial configuration for desired spectrum)                       │
│                                                                                          │
└──────────────────────────────────────────────────────────────────────────────────────────┘
"""

## Figure 5 — Physics-Informed Training Loop with Dual Loss and Backpropagation

The training loop jointly optimizes two objectives. **Path A (NLL)** pushes design parameters through the forward flow to enforce a Gaussian latent space. **Path B (Physics)** samples the latent space, pushes through the reverse flow, simulates via differentiable TMM, and penalizes spectral mismatch. Both gradient paths update all 1.2M parameters through the Adam optimizer.

In [ ]:
"""
┌─────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         PHYSICS-INFORMED TRAINING LOOP — DUAL LOSS WITH BACKPROPAGATION GRADIENT FLOW                    │
│                                                                                                          │
│   FOR epoch = 1 to 250:                                                                                  │
│     FOR each mini-batch (x_batch ∈ ℝ^(2048×20), y_batch ∈ ℝ^(2048×1000)):                               │
│                                                                                                          │
│  ╔════════════════════════════════════════════════════════════════════════════════════════════════════╗    │
│  ║  PATH A — NEGATIVE LOG-LIKELIHOOD LOSS  (all 250 epochs)                                         ║    │
│  ╚════════════════════════════════════════════════════════════════════════════════════════════════════╝    │
│                                                                                                          │
│     y_batch ──────────────────────┐                                                                      │
│     (B×1000)                      │                                                                      │
│                                   ▼                                                                      │
│                        ┌─────────────────────┐                                                           │
│     x_batch ──────────►│    cINN FORWARD      │                                                           │
│     (B×20)             │    ──────────────     │◄── cond from Encoder                                     │
│                        │    8 coupling blocks  │                                                           │
│                        │    split→subnet→      │                                                           │
│                        │    affine→flip        │                                                           │
│                        └──────────┬────────────┘                                                          │
│                                   │                                                                      │
│                          ┌────────┴────────┐                                                             │
│                          │                 │                                                              │
│                    z ∈ ℝ^(B×20)    log_det_J ∈ ℝ^B                                                       │
│                          │                 │                                                              │
│                          ▼                 ▼                                                              │
│                    ┌────────────────────────────────┐                                                     │
│                    │         L_NLL                   │                                                     │
│                    │                                │                                                     │
│                    │  = mean( 0.5·Σ(z²) − log|J| ) │                                                     │
│                    │                                │                                                     │
│                    │  Purpose: map x → z ~ N(0,I)   │                                                     │
│                    └───────────────┬────────────────┘                                                     │
│                                   │                                                                      │
│  ╔════════════════════════════════╪═══════════════════════════════════════════════════════════════════╗    │
│  ║  PATH B — PHYSICS LOSS  (epochs 21–250 only, λ = 0.1, subsample = 64)                            ║    │
│  ╚════════════════════════════════╪═══════════════════════════════════════════════════════════════════╝    │
│                                   │                                                                      │
│     z_sample ~ N(0,I) ∈ ℝ^(64×20)│              y_sub = y_batch[random 64]                               │
│        │                          │                      │                                                │
│        │                          │                      │                                                │
│        ▼                          │                      ▼                                                │
│     ┌──────────────────────┐      │           ┌─────────────────────┐                                     │
│     │   cINN REVERSE       │      │           │   SPECTRUM ENCODER   │                                     │
│     │   ──────────────     │      │           │   1000→256→256→128   │                                     │
│     │   8 blocks reversed: │      │           └──────────┬──────────┘                                     │
│     │   un-flip → inverse  │◄─────┼──── cond ────────────┘                                                │
│     │   affine coupling    │      │                                                                       │
│     └──────────┬───────────┘      │                                                                       │
│                │                  │                                                                       │
│         x_gen ∈ ℝ^(64×20)        │     (generated design parameters, differentiable)                     │
│                │                  │                                                                       │
│                ▼                  │                                                                       │
│     ┌──────────────────────────┐  │                                                                       │
│     │  DIFFERENTIABLE TMM       │  │                                                                       │
│     │  ────────────────────     │  │                                                                       │
│     │  1. Inverse scaler:       │  │                                                                       │
│     │     x_phys = x_gen·σ + μ  │  │                                                                       │
│     │                           │  │                                                                       │
│     │  2. Soft-clamp to bounds: │  │                                                                       │
│     │     σ(k·(x−lo)/(hi−lo))  │  │                                                                       │
│     │     (differentiable,      │  │                                                                       │
│     │      non-zero gradient    │  │                                                                       │
│     │      at boundaries)       │  │                                                                       │
│     │                           │  │                                                                       │
│     │  3. Dynamic E constraint: │  │                                                                       │
│     │     E ∈ [1e7(1+η),        │  │                                                                       │
│     │         1e8(1+η)]         │  │                                                                       │
│     │                           │  │                                                                       │
│     │  4. 10-layer TMM:         │  │                                                                       │
│     │     Z_i, k_i → T_total    │  │                                                                       │
│     │     → α(f) at 1000 freq   │  │                                                                       │
│     └──────────┬───────────────┘  │                                                                       │
│                │                  │                                                                       │
│         α_pred ∈ ℝ^(64×1000)     │                                                                       │
│                │                  │                                                                       │
│                ▼                  │                                                                       │
│     ┌──────────────────────────┐  │                                                                       │
│     │        L_physics          │  │                                                                       │
│     │                           │  │                                                                       │
│     │  = MSE(α_pred, y_sub)    │  │                                                                       │
│     │                           │  │                                                                       │
│     │  Purpose: generated       │  │                                                                       │
│     │  params must produce      │  │                                                                       │
│     │  valid spectra            │  │                                                                       │
│     └──────────┬───────────────┘  │                                                                       │
│                │                  │                                                                       │
│  ╔═════════════╪══════════════════╪═══════════════════════════════════════════════════════════════════╗    │
│  ║  COMBINED LOSS + BACKPROPAGATION                                                                  ║    │
│  ╚═════════════╪══════════════════╪═══════════════════════════════════════════════════════════════════╝    │
│                │                  │                                                                       │
│                └────────┬─────────┘                                                                       │
│                         ▼                                                                                │
│             ┌──────────────────────────────────┐                                                          │
│             │  L_total = L_NLL + λ · L_physics  │                                                          │
│             │                                   │                                                          │
│             │  λ = 0.0  for epochs 1–20         │                                                          │
│             │  λ = 0.1  for epochs 21–250       │                                                          │
│             └──────────────┬───────────────────┘                                                          │
│                            │                                                                              │
│                     L_total.backward()                                                                    │
│                            │                                                                              │
│          ┌─────────────────┴─────────────────┐                                                            │
│          │         GRADIENT FLOW              │                                                            │
│          │                                    │                                                            │
│          │  FROM L_NLL:                       │                                                            │
│          │    ∂L/∂z → ∂z/∂(SubNet_W)         │                                                            │
│          │    ∂L/∂log|J| → ∂|J|/∂(SubNet_W)  │                                                            │
│          │    → via cond → ∂/∂(Encoder_W)     │                                                            │
│          │                                    │                                                            │
│          │  FROM L_physics:                   │                                                            │
│          │    ∂L/∂α_pred                      │                                                            │
│          │    → ∂TMM/∂x_gen                   │                                                            │
│          │    → ∂reverse/∂(SubNet_W)          │                                                            │
│          │    → via cond → ∂/∂(Encoder_W)     │                                                            │
│          │                                    │                                                            │
│          │  Note: z_sample is NOT learnable.  │                                                            │
│          │  Gradients flow through the        │                                                            │
│          │  reverse network, not through z.   │                                                            │
│          └─────────────────┬─────────────────┘                                                            │
│                            │                                                                              │
│                            ▼                                                                              │
│             ┌──────────────────────────────┐                                                               │
│             │  GRADIENT CLIPPING            │                                                               │
│             │  clip_grad_norm_(θ, max=1.0)  │                                                               │
│             └──────────────┬───────────────┘                                                               │
│                            │                                                                              │
│                            ▼                                                                              │
│          ┌──────────────────────────────────────────┐                                                     │
│          │  ADAM OPTIMIZER STEP                      │                                                     │
│          │  ──────────────────                      │                                                     │
│          │  Updates ALL 1,207,072 parameters:       │                                                     │
│          │                                          │                                                     │
│          │  ┌─────────────────────────────────────┐ │                                                     │
│          │  │  Spectrum Encoder (354,944 params)  │ │                                                     │
│          │  │    Linear 1000→256  (256,256)       │ │                                                     │
│          │  │    Linear 256→256   (65,792)        │ │                                                     │
│          │  │    Linear 256→128   (32,896)        │ │                                                     │
│          │  └─────────────────────────────────────┘ │                                                     │
│          │                                          │                                                     │
│          │  ┌─────────────────────────────────────┐ │                                                     │
│          │  │  8 Coupling SubNets (852,128 params)│ │                                                     │
│          │  │    SubNet 1: 138→256→256→20          │ │                                                     │
│          │  │    SubNet 2: 138→256→256→20          │ │                                                     │
│          │  │    SubNet 3: 138→256→256→20          │ │                                                     │
│          │  │    SubNet 4: 138→256→256→20          │ │                                                     │
│          │  │    SubNet 5: 138→256→256→20          │ │                                                     │
│          │  │    SubNet 6: 138→256→256→20          │ │                                                     │
│          │  │    SubNet 7: 138→256→256→20          │ │                                                     │
│          │  │    SubNet 8: 138→256→256→20          │ │                                                     │
│          │  └─────────────────────────────────────┘ │                                                     │
│          │                                          │                                                     │
│          │  lr_scheduler: CosineAnnealing(T=250)    │                                                     │
│          │  weight_decay: 1e-5                      │                                                     │
│          │  optimizer.zero_grad(set_to_none=True)   │                                                     │
│          └──────────────────────────────────────────┘                                                     │
│                                                                                                          │
└─────────────────────────────────────────────────────────────────────────────────────────────────────────┘
"""

## Figure 6 — Differentiable Transfer Matrix Method (TMM) Pipeline

The TMM computes acoustic absorption for a 10-layer metamaterial stack. All operations use pure PyTorch (GPU-accelerated) with `soft_clamp` replacing `torch.clamp` to ensure non-zero gradients at parameter boundaries.

In [ ]:
"""
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│              DIFFERENTIABLE TMM — 10-LAYER ACOUSTIC METAMATERIAL STACK                   │
│                                                                                          │
│   x_gen ∈ ℝ^(B×20)   (from cINN reverse pass, differentiable)                           │
│       │                                                                                  │
│       │                                                                                  │
│   ┌───┴──────────────────────────────────────────────────────────────────────────────┐   │
│   │  STEP 1: INVERSE STANDARD SCALER                                                 │   │
│   │                                                                                   │   │
│   │  x_phys = x_gen · σ_train + μ_train                                              │   │
│   │                                                                                   │   │
│   │  (σ_train, μ_train are frozen constants from training set StandardScaler)         │   │
│   └───┬──────────────────────────────────────────────────────────────────────────────┘   │
│       │                                                                                  │
│       ▼                                                                                  │
│   ┌──────────────────────────────────────────────────────────────────────────────────┐   │
│   │  STEP 2: SOFT CLAMP TO PHYSICAL BOUNDS                                           │   │
│   │                                                                                   │   │
│   │  soft_clamp(x, lo, hi) = lo + (hi − lo) · σ( k · (x − lo) / (hi − lo) )        │   │
│   │                                          where k = 10 (sharpness)                 │   │
│   │                                                                                   │   │
│   │  ┌────────────────────────────────────────────────────────────────────────────┐   │   │
│   │  │                                                                            │   │   │
│   │  │   hard clamp:    ─────┐              ┌──────   (zero gradient at bounds)   │   │   │
│   │  │                       │              │                                     │   │   │
│   │  │   soft clamp:   ╱─────────────────────╲       (smooth, always non-zero      │   │   │
│   │  │                ╱                       ╲       gradient everywhere)          │   │   │
│   │  │              lo                       hi                                   │   │   │
│   │  └────────────────────────────────────────────────────────────────────────────┘   │   │
│   │                                                                                   │   │
│   │  Parameter groups and their bounds:                                               │   │
│   │                                                                                   │   │
│   │    Index  │ Symbol    │ Description                │ Lower        │ Upper          │   │
│   │    ───────┼───────────┼────────────────────────────┼──────────────┼────────────────│   │
│   │    0–9    │ d₁ – d₁₀  │ Layer thicknesses (m)      │ 1×10⁻³       │ 20×10⁻³       │   │
│   │    10–15  │ m₂,₃,₅,₆, │ Hollow cavity dims (m)     │ 20×10⁻³      │ 1980×10⁻³     │   │
│   │           │ m₈,m₉     │                            │              │                │   │
│   │    16     │ ρ          │ Density (kg/m³)            │ 1000         │ 1500           │   │
│   │    17     │ η          │ Loss factor                │ 0.1          │ 0.8            │   │
│   │    18     │ E          │ Young's modulus (Pa)       │ 1e7·(1+η)    │ 1e8·(1+η)     │   │
│   │    19     │ ν          │ Poisson's ratio            │ 0.4          │ 0.49           │   │
│   │                                                                                   │   │
│   └───┬──────────────────────────────────────────────────────────────────────────────┘   │
│       │                                                                                  │
│       ▼                                                                                  │
│   ┌──────────────────────────────────────────────────────────────────────────────────┐   │
│   │  STEP 3: 10-LAYER TRANSFER MATRIX METHOD                                         │   │
│   │                                                                                   │   │
│   │   Water (ρ_w, c_w)                                                                │   │
│   │       │                                                                           │   │
│   │       ▼                                                                           │   │
│   │   ┌───────────┐                                                                  │   │
│   │   │  Layer 1   │  d₁, ρ, E_eff(E,ν,η), hollow cavity (m₂ if applicable)         │   │
│   │   │  T₁ = [A₁ B₁; C₁ D₁]                                                       │   │
│   │   └─────┬─────┘                                                                  │   │
│   │         ▼                                                                         │   │
│   │   ┌───────────┐                                                                  │   │
│   │   │  Layer 2   │  d₂, same material properties, cavity dim m₂                    │   │
│   │   │  T₂ = [A₂ B₂; C₂ D₂]                                                       │   │
│   │   └─────┬─────┘                                                                  │   │
│   │         ⋮                                                                         │   │
│   │   ┌───────────┐                                                                  │   │
│   │   │  Layer 10  │  d₁₀                                                            │   │
│   │   │  T₁₀ = [A₁₀ B₁₀; C₁₀ D₁₀]                                                 │   │
│   │   └─────┬─────┘                                                                  │   │
│   │         ▼                                                                         │   │
│   │   Steel backing (rigid)                                                           │   │
│   │                                                                                   │   │
│   │   T_total = T₁ · T₂ · T₃ · … · T₁₀    (2×2 complex matrix product)              │   │
│   │                                                                                   │   │
│   │   For each frequency f ∈ {f₁, f₂, …, f₁₀₀₀}  (200–2000 Hz):                    │   │
│   │     • Compute wavenumber k_i = 2πf / c_eff                                       │   │
│   │     • Compute impedance Z_i = ρ_eff · c_eff                                      │   │
│   │     • Build transfer matrix T_i(f)                                                │   │
│   │     • T_total(f) = ∏ T_i(f)                                                      │   │
│   │     • Reflection R(f) = (Z_in − Z_w) / (Z_in + Z_w)                              │   │
│   │     • Absorption α(f) = 1 − |R(f)|²                                              │   │
│   │                                                                                   │   │
│   │   All operations: torch.matmul, torch.exp, torch.abs, torch.sqrt(·+ε)            │   │
│   │   → fully differentiable, GPU-accelerated                                         │   │
│   │                                                                                   │   │
│   └───┬──────────────────────────────────────────────────────────────────────────────┘   │
│       │                                                                                  │
│       ▼                                                                                  │
│   α_pred ∈ ℝ^(B×1000)                                                                   │
│   (predicted absorption spectrum, differentiable w.r.t. x_gen)                           │
│                                                                                          │
└──────────────────────────────────────────────────────────────────────────────────────────┘
"""

## Figure 7 — Training Schedule and Hyperparameter Summary

In [ ]:
"""
┌──────────────────────────────────────────────────────────────────────────────────────────┐
│                           TRAINING SCHEDULE                                               │
│                                                                                          │
│   Epoch:  1          20         21                                           250         │
│           │          │          │                                             │           │
│           ▼          ▼          ▼                                             ▼           │
│   ┌──────────────────────┬───────────────────────────────────────────────────────────┐   │
│   │     WARMUP PHASE     │              PHYSICS-INFORMED PHASE                       │   │
│   │     (20 epochs)      │              (230 epochs)                                 │   │
│   │                      │                                                           │   │
│   │  L = L_NLL           │  L = L_NLL + 0.1 · L_physics                             │   │
│   │                      │                                                           │   │
│   │  Pure normalizing    │  Dual-objective optimization:                             │   │
│   │  flow training.      │  Gaussian latent space +                                  │   │
│   │  Stabilizes latent   │  physically valid generated                               │   │
│   │  space mapping       │  parameters                                               │   │
│   │  before adding       │                                                           │   │
│   │  physics constraint  │  Physics subsample: 64 per batch                          │   │
│   │                      │  (3.1% of batch_size=2048)                                │   │
│   └──────────────────────┴───────────────────────────────────────────────────────────┘   │
│                                                                                          │
│   Learning Rate (CosineAnnealing):                                                       │
│                                                                                          │
│   lr  1e-3 ┐                                                                             │
│            │╲                                                                            │
│            │  ╲                                                                          │
│            │    ╲                                                                        │
│            │      ╲                                                                      │
│            │        ╲  ╱ cosine decay                                                    │
│            │         ╲╱                                                                  │
│   lr  ~0   └─────────────────────────────────────────────────────────────────────        │
│            1                                                          250                 │
│                                                                                          │
│─────────────────────────────────────────────────────────────────────────────────────────│
│                                                                                          │
│                      HYPERPARAMETER SUMMARY                                              │
│                                                                                          │
│   ┌─────────────────────────┬───────────────────────────────────────┐                    │
│   │  Hyperparameter         │ Value                                 │                    │
│   │  ───────────────────────┼─────────────────────────────────────  │                    │
│   │  Epochs                 │ 250                                   │                    │
│   │  Batch size             │ 2048                                  │                    │
│   │  Optimizer              │ Adam                                  │                    │
│   │  Learning rate          │ 1×10⁻³                                │                    │
│   │  Weight decay           │ 1×10⁻⁵                                │                    │
│   │  LR scheduler           │ CosineAnnealingLR (T_max=250)        │                    │
│   │  Gradient clipping      │ max_norm = 1.0                       │                    │
│   │  Physics loss weight λ  │ 0.1                                  │                    │
│   │  Physics warmup         │ 20 epochs (NLL only)                 │                    │
│   │  Physics subsample      │ 64 / batch                           │                    │
│   │  Coupling blocks        │ 8                                    │                    │
│   │  Encoder output dim     │ 128                                  │                    │
│   │  Hidden dim (all nets)  │ 256                                  │                    │
│   │  Activation             │ LeakyReLU(0.01)                      │                    │
│   │  Clamp range (s)        │ [−3, 3]  (soft_clamp)                │                    │
│   │  Soft clamp sharpness   │ k = 10                               │                    │
│   │  Latent distribution    │ N(0, I₂₀)                            │                    │
│   └─────────────────────────┴───────────────────────────────────────┘                    │
│                                                                                          │
│─────────────────────────────────────────────────────────────────────────────────────────│
│                                                                                          │
│                         DATASET SUMMARY                                                  │
│                                                                                          │
│   ┌─────────────────────────┬───────────────────────────────────────┐                    │
│   │  Property               │ Value                                 │                    │
│   │  ───────────────────────┼─────────────────────────────────────  │                    │
│   │  Total samples          │ 1,000,000 (Latin Hypercube Sampling) │                    │
│   │  Training set           │ 800,000  (80%)                       │                    │
│   │  Validation set         │ 100,000  (10%)                       │                    │
│   │  Test set               │ 100,000  (10%)                       │                    │
│   │  Input (x)              │ 20 design parameters (scaled)        │                    │
│   │  Output (y)             │ 1000-point absorption spectrum       │                    │
│   │  Frequency range        │ 200–2000 Hz                          │                    │
│   │  Scaling                │ StandardScaler (per-feature)         │                    │
│   │  Precision              │ float32                              │                    │
│   └─────────────────────────┴───────────────────────────────────────┘                    │
│                                                                                          │
└──────────────────────────────────────────────────────────────────────────────────────────┘
"""

## Design Decision Summary

| Design Choice | Rationale |
|---|---|
| **cINN (normalizing flow)** | Exact density estimation; bijective mapping enables both forward (x → z) and reverse (z → x) passes from a single trained model. |
| **8 coupling blocks** | Each 10-dim half is transformed 4 times. For 20-dim parameter space, this provides sufficient expressivity while keeping computation tractable. |
| **Condition on full spectrum** | 1000-point spectrum input (not downsampled) preserves fine spectral features critical for acoustic design. Encoder compresses to 128-dim. |
| **Physics-informed loss** | L_physics forces generated parameters to produce valid spectra when simulated, not just satisfy latent-space Gaussianity. |
| **Soft clamp (sigmoid)** | Unlike `torch.clamp` (zero gradient at bounds), sigmoid-based soft clamp provides non-zero gradients everywhere, enabling gradient flow through the TMM. |
| **Warmup (20 epochs)** | Pure NLL training first stabilizes the latent-space mapping before the physics loss introduces a competing objective. |
| **Physics subsample = 64** | TMM is ~30× more expensive than a forward cINN pass. Running TMM on 3.1% of the batch keeps per-step cost manageable (~1.3×). |
| **λ = 0.1** | Balances NLL (~5–10 range) against physics MSE (~0.01–0.1) to prevent either objective from dominating. |
| **CosineAnnealing LR** | Smooth decay avoids sudden learning rate drops; allows continued fine-tuning in later epochs. |
| **Gradient clipping = 1.0** | Prevents gradient explosion from TMM backpropagation (complex matrix chain of 10 layers). |

---

### Key Equations

$$\mathcal{L}_{NLL} = \mathbb{E}\left[ \frac{1}{2} \|z\|^2 - \log |\det J| \right]$$

$$\mathcal{L}_{physics} = \text{MSE}\left( \alpha_{TMM}(G^{-1}(z; y)), \; y_{target} \right)$$

$$\mathcal{L}_{total} = \mathcal{L}_{NLL} + \lambda \cdot \mathcal{L}_{physics}$$

where $G$ is the forward flow, $G^{-1}$ is the reverse flow, and $\alpha_{TMM}$ is the differentiable Transfer Matrix Method.

---

*This architecture reference is intended for replication in conference paper figures. All dimensions, parameter counts, and data flow directions are exact.*